#### Kube-vip Endpoint 

In [ ]:
VIP="172.16.6.85"
INTERFACE="ens192"

KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases | jq -r ".[0].name")

alias kube-vip="ctr image pull ghcr.io/kube-vip/kube-vip:$KVVERSION; ctr run --rm --net-host ghcr.io/kube-vip/kube-vip:$KVVERSION vip /kube-vip"

mkdir -p /etc/kubernetes/manifests/

kube-vip manifest pod \
    --interface $INTERFACE \
    --address $VIP \
    --controlplane \
    --services \
    --arp \
    --leaderElection | tee /etc/kubernetes/manifests/kube-vip.yaml

ctr images ls | grep kube-vip


kubeadm init --control-plane-endpoint "172.16.6.85:6443" --upload-certs --pod-network-cidr=10.244.0.0/16 --apiserver-cert-extra-sans=172.16.6.85


mkdir -p $HOME/.kube
sudo cp -i /etc/kubernetes/admin.conf $HOME/.kube/config
sudo chown $(id -u):$(id -g) $HOME/.kube/config

---

#### Script

In [ ]:
cat > kube-vip.sh << 'OF'
#!/bin/bash
# =============================================================================
# Kube-VIP Static Pod Setup Script
# Purpose: Deploy Kube-VIP as a static pod for HA Kubernetes control plane
# Author: Generated for Khaled
# =============================================================================

set -euo pipefail  # Best practice: exit on error, undefined vars, pipe failures

# ========================= CONFIGURATION =========================
VIP="172.16.6.85"
INTERFACE="ens192"
POD_NETWORK_CIDR="10.244.0.0/16"

# Colors for better output
RED='\033[0;31m'
GREEN='\033[0;32m'
YELLOW='\033[1;33m'
BLUE='\033[0;34m'
NC='\033[0m' # No Color

log() { echo -e "${BLUE}[INFO]${NC} $1" }

success() { echo -e "${GREEN}[SUCCESS]${NC} $1" }

warn() { echo -e "${YELLOW}[WARN]${NC} $1" }

error() {
    echo -e "${RED}[ERROR]${NC} $1" >&2
    exit 1
}

# Check if running as root
if [[ $EUID -ne 0 ]]; then
    error "This script must be run as root"
fi

log "Starting Kube-VIP + kubeadm initialization setup..."

# =========================1. GET LATEST KUBE-VIP VERSION =========================
log "1. Fetching latest kube-vip version..."
KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases/latest | \
            jq -r '.tag_name' || echo "v0.8.0")

if [[ -z "$KVVERSION" || "$KVVERSION" == "null" ]]; then
    warn "Failed to fetch latest version, using fallback v0.8.0"
    KVVERSION="v0.8.0"
fi

success "✅ Using kube-vip version: $KVVERSION"

# =========================2. SETUP KUBE-VIP ALIAS =========================
log "2. Setting up kube-vip command..."

# Create a proper wrapper script with version embedded
cat > /usr/local/bin/kube-vip << EOF
#!/bin/bash
# Kube-VIP wrapper - Version: ${KVVERSION}

KVVERSION="${KVVERSION}"
IMAGE="ghcr.io/kube-vip/kube-vip:\${KVVERSION}"

# Pull image if not present
if ! ctr images ls | grep -q "kube-vip:\${KVVERSION}"; then
    echo "Pulling kube-vip image: \${IMAGE}" >&2
    ctr image pull "\${IMAGE}" >/dev/null 2>&1 || {
        echo "Failed to pull kube-vip image" >&2
        exit 1
    }
fi

ctr run --rm --net-host "\${IMAGE}" vip /kube-vip "\$@"
EOF

chmod +x /usr/local/bin/kube-vip

success "✅ kube-vip command installed (version ${KVVERSION})"


# =========================3. CREATE KUBE-VIP MANIFEST =========================
log "3. Generating Kube-VIP static pod manifest..."

mkdir -p /etc/kubernetes/manifests/

kube-vip manifest pod \
    --interface "$INTERFACE" \
    --address "$VIP" \
    --controlplane \
    --services \
    --arp \
    --leaderElection | tee /etc/kubernetes/manifests/kube-vip.yaml

if [[ ! -f /etc/kubernetes/manifests/kube-vip.yaml ]]; then
    error "Failed to create kube-vip manifest"
fi

success "✅ Kube-VIP manifest created at /etc/kubernetes/manifests/kube-vip.yaml"

# ========================= 4 VERIFY IMAGE =========================
log "4. Verifying kube-vip image..."
ctr images ls | grep -i kube-vip || warn "kube-vip image not found in containerd"

log "✅ Starting Kubernetes cluster initialization..."

echo ""
echo "═══════════════════════════════════════════════════════════════"
echo "                  🚀 RUNNING KUBEADM INIT"
echo "═══════════════════════════════════════════════════════════════"
echo ""

kubeadm init \
    --control-plane-endpoint "${VIP}:6443" \
    --upload-certs \
    --pod-network-cidr "$POD_NETWORK_CIDR" \
    --apiserver-cert-extra-sans "$VIP" 2>&1 | tee /var/log/kubeadm-init-$(date +%Y%m%d-%H%M).log

INIT_EXIT_CODE=${PIPESTATUS[0]}

if [[ $INIT_EXIT_CODE -eq 0 ]]; then
    success "✅ Kubernetes control plane initialized successfully!"
    
    # Extract and show join commands clearly
    echo ""
    echo "📋 IMPORTANT JOIN COMMANDS:"
    echo "───────────────────────────────────────────────────────────────"
    grep -A 5 -E "(kubeadm join|certificate-key)" /var/log/kubeadm-init-*.log | tail -n 20
    echo "───────────────────────────────────────────────────────────────"
else
    error "❌ kubeadm init failed with exit code $INIT_EXIT_CODE"
fi

# ========================= SETUP KUBECONFIG =========================
log "Setting up kubectl configuration for current user..."

mkdir -p $HOME/.kube
cp -i /etc/kubernetes/admin.conf $HOME/.kube/config
chown $(id -u):$(id -g) $HOME/.kube/config

success "✅ Kubeconfig configured successfully!"
echo "You can now run: kubectl get nodes"

# ========================= FINAL SUMMARY =========================
echo ""
echo "═══════════════════════════════════════════════════════════════"
echo "🎉 Setup Completed Successfully!"
echo "═══════════════════════════════════════════════════════════════"
echo "VIP Address      : $VIP"
echo "Interface        : $INTERFACE"
echo "Kube-VIP Version : $KVVERSION"
echo "Pod CIDR         : $POD_NETWORK_CIDR"
echo ""
echo "Next steps:"
echo "   1. Install your CNI (Flannel, Calico, Cilium, etc.)"
echo "   2. Join other control plane nodes using the join command above"
echo "   3. kubectl get pods -n kube-system -w"
echo "═══════════════════════════════════════════════════════════════"
OF

In [ ]:
chmod +x kube-vip.sh && ./kube-vip.sh